# LSTM：解决长期依赖的门控机制
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：LSTM.py | 核心功能：遗忘门/输入门/输出门、细胞状态、长距离依赖

## 概述

LSTM（Long Short-Term Memory）是 RNN 最成功的改进版本。它引入了**细胞状态**（cell state）和三个**门控机制**，解决了标准 RNN 的梯度消失问题。

三个门：遗忘门（丢弃旧信息）、输入门（写入新信息）、输出门（输出信息）。细胞状态像一条"传送带"，信息可以几乎无损地传递很远。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

## 数学原理

### 1. LSTM 的门控机制

**代码对应**：LSTM 通过三个门控制信息流，解决 RNN 的梯度消失问题。

**遗忘门**：$\mathbf{f}_t = \sigma(\mathbf{W}_f[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$

**输入门**：$\mathbf{i}_t = \sigma(\mathbf{W}_i[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i)$

**候选记忆**：$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)$

**细胞状态更新**：$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$

**输出门**：$\mathbf{o}_t = \sigma(\mathbf{W}_o[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)$

**隐藏状态**：$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$

### 2. 为什么 LSTM 能解决梯度消失

细胞状态的梯度：

$$\frac{\partial\mathbf{c}_t}{\partial\mathbf{c}_{t-1}} = \text{diag}(\mathbf{f}_t)$$

当 $\mathbf{f}_t \approx \mathbf{1}$ 时（遗忘门打开），$\frac{\partial\mathbf{c}_t}{\partial\mathbf{c}_{t-1}} \approx \mathbf{I}$，梯度沿细胞状态**无衰减**传播。这是 LSTM 的关键创新——细胞状态是一条"梯度高速公路"。

对比 RNN：$\frac{\partial\mathbf{h}_t}{\partial\mathbf{h}_{t-1}} = \text{diag}(\tanh'(\cdot))\mathbf{W}_{hh}$，连乘后必然衰减。

### 3. LSTM 的参数量

每个门的参数：$4 \times (d_h \times d_h + d_h \times d_x + d_h)$

总参数量约为普通 RNN 的 4 倍（4 个门共享相同的输入组合）。

### 1. 序列数据生成

运行 1. 序列数据生成 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
timesteps = 100

def generate_sine_data(n_samples, timesteps):
    X, y = [], []
    for _ in range(n_samples):
        start = np.random.uniform(0, 2 * np.pi)
        t = np.linspace(start, start + 4 * np.pi, timesteps + 1)
# --- 数值计算 ---
        data = np.sin(t) + 0.1 * np.random.randn(len(t))
        X.append(data[:-1])
        y.append(data[-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = generate_sine_data(500, timesteps)
X_train, X_test = torch.FloatTensor(X[:400]).unsqueeze(-1), torch.FloatTensor(X[400:]).unsqueeze(-1)
y_train, y_test = torch.FloatTensor(y[:400]), torch.FloatTensor(y[400:])

### 2. LSTM 模型

运行 2. LSTM 模型 的代码，观察算法在该环节的行为。

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
# --- 继续 ---
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
# --- 返回结果 ---
        return out.squeeze(-1)

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"=== LSTM 模型结构 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

### 3. LSTM 门控机制

运行 3. LSTM 门控机制 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== LSTM 门控机制 ===")
print("遗忘门 f_t = σ(W_f × [h_{t-1}, x_t] + b_f)  — 决定丢弃多少旧信息")
print("输入门 i_t = σ(W_i × [h_{t-1}, x_t] + b_i)  — 决定写入多少新信息")
print("候选值 C̃_t = tanh(W_C × [h_{t-1}, x_t] + b_C)")
print("细胞状态 C_t = f_t × C_{t-1} + i_t × C̃_t   — 核心：加法更新，梯度流通好")
# --- 输出结果 ---
print("输出门 o_t = σ(W_o × [h_{t-1}, x_t] + b_o)")
print("隐状态 h_t = o_t × tanh(C_t)")

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 50
for epoch in range(n_epochs):
    model.train()
    output = model(X_train)
# --- 继续 ---
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test)
            test_rmse = torch.sqrt(criterion(test_pred, y_test)).item()
# --- 输出结果 ---
        print(f"  Epoch {epoch+1:>2}: Train RMSE={torch.sqrt(loss).item():.4f}, "
              f"Test RMSE={test_rmse:.4f}")

### 5. 预测对比

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 预测效果 ===")
model.eval()
with torch.no_grad():
    preds = model(X_test).numpy()
print(f"{'真实值':>10} {'预测值':>10} {'误差':>10}")
# --- 循环处理 ---
for i in range(10):
    print(f"{y_test[i].item():>10.4f} {preds[i]:>10.4f} {abs(y_test[i].item()-preds[i]):>10.4f}")

### 6. LSTM vs 简单 RNN 对比

对比不同模型或配置的性能差异。

In [ ]:
from torch.nn import RNN as SimpleRNN

class SimpleRNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = SimpleRNN(1, 64, num_layers=2, batch_first=True)
        self.fc = nn.Linear(64, 1)
# --- 定义函数 ---
    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 64)
        out, _ = self.rnn(x, h0)
        return self.fc(out[:, -1, :]).squeeze(-1)

print("\n=== LSTM vs RNN 训练对比 ===")
for name, Model in [("RNN", SimpleRNNModel), ("LSTM", LSTMModel)]:
    m = Model()
    opt = optim.Adam(m.parameters(), lr=0.001)
    for _ in range(50):
# --- 继续 ---
        m.train()
        out = m(X_train)
        loss = criterion(out, y_train)
        opt.zero_grad()
        loss.backward()
# --- 继续 ---
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()
    m.eval()
    with torch.no_grad():
        rmse = torch.sqrt(criterion(m(X_test), y_test)).item()
# --- 输出结果 ---
    print(f"  {name}: Test RMSE={rmse:.4f}")

print("\n=== LSTM 要点 ===")
print("- 解决了 RNN 的梯度消失问题，可学习长距离依赖")
print("- 细胞状态 C_t 通过加法更新（而非乘法），梯度流通顺畅")
print("- 参数量约为 RNN 的 4 倍（3 个门 + 候选值）")
print("- 适合：机器翻译、文本生成、语音识别、时间序列预测")

## 关键代码解释

### LSTM 单元

```python
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
output, (hn, cn) = lstm(x)  # cn 是细胞状态
```

### 门控机制

```python
f_t = sigmoid(W_f @ [h_{t-1}, x_t])  # 遗忘门：丢多少旧信息
i_t = sigmoid(W_i @ [h_{t-1}, x_t])  # 输入门：加多少新信息
o_t = sigmoid(W_o @ [h_{t-1}, x_t])  # 输出门：输出什么
```

## 注意事项

1. **比 RNN 慢**：参数量是 RNN 的 4 倍
2. **仍然有梯度问题**：比 RNN 好很多但不是完全解决
3. **不能并行**：时间步仍需串行

## 总结与延伸

以上代码展示了 **LSTM** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **GRU**：LSTM 的简化版，2 个门，参数更少
- **Peephole LSTM**：让门也能看到细胞状态
- **Transformer**：完全抛弃循环结构，用自注意力替代
